# TP Jour 1 — Telco Customer Churn

**Contexte** : un opérateur télécom veut anticiper les clients qui vont résilier leur contrat (`Churn`). On a 7043 clients et 20 variables descriptives. Cible : binaire (Yes/No), classe minoritaire ~26%.

**Objectif pédagogique** : appliquer la chaîne complète vue en cours — audit qualité, EDA, préparation sans fuite, comparaison de 6 algos, métriques adaptées au déséquilibre, tuning d'hyperparamètres, interprétation.

**Règle absolue** : aucune transformation appliquée avant le `train_test_split`. Tout passe par `Pipeline` + `ColumnTransformer`.


## 0. Installation

Exécutez la cellule suivante en premier. Elle installe automatiquement les paquets manquants (utile sur un Jupyter/Colab vierge).

In [1]:
# Installation des dépendances (exécutez UNE FOIS, puis passez à la suite)
# Compatible Jupyter classique, JupyterLite/Pyodide, XPython/mambajs, Google Colab.
%pip install seaborn scikit-learn pandas matplotlib scipy


  Using cached threadpoolctl-3.6.0-py3-none-any.whl.metadata (13 kB)
  Using cached cycler-0.12.1-py3-none-any.whl.metadata (3.8 kB)
   ---------------------------------------- 0.0/8.1 MB ? eta -:--:--
   ---------- ----------------------------- 2.1/8.1 MB 11.0 MB/s eta 0:00:01
   -------------------- ------------------- 4.2/8.1 MB 10.6 MB/s eta 0:00:01
   ------------------------------- -------- 6.3/8.1 MB 10.3 MB/s eta 0:00:01
   ---------------------------------------- 8.1/8.1 MB 9.8 MB/s  0:00:00
   ---------------------------------------- 0.0/9.9 MB ? eta -:--:--
   -------- ------------------------------- 2.1/9.9 MB 10.4 MB/s eta 0:00:01
   ------------------ --------------------- 4.5/9.9 MB 10.7 MB/s eta 0:00:01
   ---------------------------- ----------- 7.1/9.9 MB 11.5 MB/s eta 0:00:01
   -------------------------------------- - 9.4/9.9 MB 11.4 MB/s eta 0:00:01
   ---------------------------------------- 9.9/9.9 MB 10.7 MB/s  0:00:00
   ----------------------------------------

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.

[notice] A new release of pip is available: 25.3 -> 26.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## 1. Audit qualité

**Q1.1** Affichez `info()` et `describe()`. Identifiez les colonnes par type.

In [2]:
# Imports — exécutez cette cellule
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score, RandomizedSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (confusion_matrix, classification_report,
                             roc_auc_score, average_precision_score,
                             roc_curve, precision_recall_curve, f1_score)

RNG = 42
sns.set_theme(style='whitegrid')
pd.set_option('display.max_columns', 30)


In [3]:
URL = 'https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv'
df = pd.read_csv(URL)
print(df.shape)
df.head(3)


(7043, 21)


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes


In [4]:
df.info()
df.describe(include='all').T


<class 'pandas.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   str    
 1   gender            7043 non-null   str    
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   str    
 4   Dependents        7043 non-null   str    
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   str    
 7   MultipleLines     7043 non-null   str    
 8   InternetService   7043 non-null   str    
 9   OnlineSecurity    7043 non-null   str    
 10  OnlineBackup      7043 non-null   str    
 11  DeviceProtection  7043 non-null   str    
 12  TechSupport       7043 non-null   str    
 13  StreamingTV       7043 non-null   str    
 14  StreamingMovies   7043 non-null   str    
 15  Contract          7043 non-null   str    
 16  PaperlessBilling  7043 non-null   str    
 17  Paymen

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
customerID,7043,7043,7590-VHVEG,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
gender,7043,2,Male,3555,NaN,NaN,NaN,NaN,NaN,NaN,NaN
SeniorCitizen,7043.0,NaN,NaN,NaN,0.162147,0.368612,0.0,0.0,0.0,0.0,1.0
Partner,7043,2,No,3641,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Dependents,7043,2,No,4933,NaN,NaN,NaN,NaN,NaN,NaN,NaN
tenure,7043.0,NaN,NaN,NaN,32.371149,24.559481,0.0,9.0,29.0,55.0,72.0
PhoneService,7043,2,Yes,6361,NaN,NaN,NaN,NaN,NaN,NaN,NaN
MultipleLines,7043,3,No,3390,NaN,NaN,NaN,NaN,NaN,NaN,NaN
InternetService,7043,3,Fiber optic,3096,NaN,NaN,NaN,NaN,NaN,NaN,NaN
OnlineSecurity,7043,3,No,3498,NaN,NaN,NaN,NaN,NaN,NaN,NaN


**Q1.2** La colonne `TotalCharges` est en `object`. Pourquoi ? Convertissez-la en numérique en gérant les valeurs problématiques (`pd.to_numeric(..., errors='coerce')`). Combien de NaN apparaissent ?

In [27]:
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
print(f"NaN créés : {df['TotalCharges'].isna().sum()}")


NaN créés : 11


**Q1.3** Inspectez les NaN ainsi créés : à quel sous-groupe correspondent-ils ? (indice : regardez la colonne `tenure` pour ces lignes). C'est un cas typique de **manquant déguisé** : le NaN n'est pas aléatoire, il a une signification métier.

In [ ]:
mask_na = df['TotalCharges'].isna()
print(df.loc[mask_na, ['tenure', 'MonthlyCharges','Churn']].describe())
df.loc[mask_na, 'Totalcharges'] = 0

       tenure  MonthlyCharges
count    11.0       11.000000
mean      0.0       41.418182
std       0.0       23.831484
min       0.0       19.700000
25%       0.0       20.125000
50%       0.0       25.750000
75%       0.0       58.975000
max       0.0       80.850000
7043


**Q1.4** Affichez le taux de doublons et la distribution de la cible `Churn`. Le dataset est-il équilibré ?

In [34]:
print(f"Doublons : {df.duplicated().sum()}")
print(df['Churn'].value_counts(normalize=True).round(3))

Doublons : 0
Churn
No     0.735
Yes    0.265
Name: proportion, dtype: float64


## 2. Analyse exploratoire (EDA)

**Q2.1** Tracez les distributions de `tenure`, `MonthlyCharges`, `TotalCharges` (3 subplots, KDE ou histogramme). Que remarquez-vous sur la forme de `tenure` ?

In [8]:
# TODO : 3 subplots côte à côte


**Q2.2** Pour les variables catégorielles `Contract`, `PaymentMethod`, `InternetService`, calculez le **taux de churn par modalité**. Quelles modalités prédisent le mieux le churn ?

In [9]:
# TODO : groupby + mean sur churn binaire


**Q2.3** Tracez un boxplot de `MonthlyCharges` par `Churn`. Visuellement, le churn est-il associé à des charges mensuelles plus élevées ?

In [10]:
# TODO


**Q2.4** Calculez la matrice de corrélation entre les 3 variables numériques (`tenure`, `MonthlyCharges`, `TotalCharges`) et tracez-la (heatmap). Identifiez la corrélation la plus forte et expliquez-la métier.

In [11]:
# TODO


**Q2.5 (outliers)** Sur `MonthlyCharges` et `TotalCharges`, calculez les bornes IQR (Q1 − 1.5·IQR, Q3 + 1.5·IQR) et le pourcentage de points hors bornes. Décision : suppression, capping, ou rien ? Justifiez.

In [12]:
# TODO


## 3. Préparation

**Q3.1** Créez la cible binaire `y` (1 si Churn == 'Yes', sinon 0) et la matrice `X` en retirant `customerID` et `Churn`.

In [13]:
# TODO


**Q3.2** Séparez les listes de colonnes `num_cols` (numériques) et `cat_cols` (catégorielles). Vérifiez que `TotalCharges` est bien dans `num_cols`.

In [14]:
# TODO


**Q3.3** Faites un split train/test **stratifié**, 20% de test, `random_state=RNG`. Vérifiez que la proportion de churn est conservée dans les deux sous-ensembles.

In [15]:
# TODO


**Q3.4** Construisez le `ColumnTransformer` :
- numériques → `SimpleImputer(strategy='median')` puis `StandardScaler()`
- catégorielles → `SimpleImputer(strategy='most_frequent')` puis `OneHotEncoder(handle_unknown='ignore')`

Puis enchaînez-le avec un classifieur dans un `Pipeline`. C'est ce pipeline qu'on va `fit` — pas les données préparées séparément.

In [16]:
# TODO


## 4. Comparaison de modèles

**Q4.1** Comparez **6 classifieurs** par validation croisée stratifiée 5-fold sur le train, métrique = ROC AUC :
- `KNeighborsClassifier(n_neighbors=15)`
- `GaussianNB()`
- `LogisticRegression(max_iter=1000, class_weight='balanced')`
- `DecisionTreeClassifier(max_depth=5, random_state=RNG)`
- `RandomForestClassifier(n_estimators=200, random_state=RNG, n_jobs=-1)`
- `GradientBoostingClassifier(random_state=RNG)`

Pour chaque modèle, affichez moyenne ± écart-type.

In [17]:
# TODO : boucle sur dict de modèles


**Q4.2** Quel modèle gagne ? L'écart est-il significatif (>1 écart-type) ? Quel modèle a la plus grande variance entre folds — pourquoi ?

*Réponse libre :*

## 5. Évaluation détaillée

**Q5.1** Entraînez le meilleur modèle sur tout le train et prédisez sur le test. Calculez : accuracy, F1, ROC AUC, PR AUC. Pourquoi accuracy seule serait trompeuse ici ?

In [18]:
# TODO


**Q5.2** Tracez la matrice de confusion. Combien de churners avez-vous ratés (faux négatifs) ? Combien de fausses alertes (faux positifs) ?

In [19]:
# TODO


**Q5.3** Tracez les courbes ROC et Precision-Recall sur le test. Laquelle est la plus informative pour ce problème déséquilibré ?

In [20]:
# TODO


**Q5.4 (déséquilibre — threshold tuning)** Le seuil 0.5 par défaut est arbitraire. Calculez `precision_recall_curve` puis trouvez le seuil qui maximise le F1. Comparez le F1 au seuil 0.5 vs au seuil optimisé.

*Indice : `f1_scores = 2*p*r/(p+r+1e-9)`, prendre `argmax`.*

In [21]:
# TODO


## 6. Tuning d'hyperparamètres

**Q6.1** Sur le pipeline `RandomForest`, lancez un `RandomizedSearchCV` (n_iter=15, cv=5, scoring='roc_auc') sur :
- `clf__n_estimators` : [100, 200, 400]
- `clf__max_depth` : [None, 5, 10, 20]
- `clf__min_samples_leaf` : [1, 5, 20]
- `clf__max_features` : ['sqrt', 'log2', 0.5]

Affichez les meilleurs paramètres et le score de validation.

In [22]:
# TODO


**Q6.2** Évaluez le modèle tuné sur le test. Le gain par rapport au modèle par défaut justifie-t-il le coût de calcul ?

In [23]:
# TODO


## 7. Interprétation

**Q7.1** Sur la régression logistique tunée (avec `class_weight='balanced'`), récupérez les coefficients après pipeline. Quelles sont les 5 variables les plus prédictives du churn (positives) et anti-churn (négatives) ?

*Indice : `pipe.named_steps['preprocess'].get_feature_names_out()` pour récupérer les noms après OneHot.*

In [24]:
# TODO


**Q7.2** Sur la `RandomForest` tunée, affichez les 10 variables les plus importantes (`feature_importances_`). Comparez avec le top de la régression logistique : convergence ou divergence ?

In [25]:
# TODO


## 8. Pour aller plus loin (optionnel)

- **Calibration** : la `RandomForest` produit-elle des probabilités calibrées ? Tracez une courbe de calibration (`from sklearn.calibration import calibration_curve`).
- **SHAP** : utilisez `shap` pour expliquer une prédiction individuelle.
- **Coût métier** : si rater un churner coûte 10× plus qu'une fausse alerte, quel seuil choisir ? Recalculez le F-beta avec β=2.